# AI Projects Growth Rate

## Table of Contents
- [Introduction](#introduction)
- [Database Connection](#connect-to-the-augur-database)
- [Load the Repositories](#load-the-urls-of-ai-repositories)
- [Retrieve Repository IDs](#retrieve-the-repository-ids-and-the-repository-names)
- [Fetch the Contribution Data](#fetch-the-contribution-data)
  - [Pull Request Contribution Data](#pull-request-contribution-data)
  - [Commit Contribution Data](#commit-contribution-data)
  - [Issue Contribution Data](#issue-contribution-data)
  - [Pull Request Review Contribution Data](#pull-request-review-contribution-data)
  - [Message Contribution Data](#message-contribution-data)
- [Calculate the Age of Projects](#calculate-the-age-of-projects)
- [Growth Rate Calculation](#growth-rate-calculation)
  - [Growth Rate Plots](#growth-rate-analysis---plots)

## Introduction

In this notebook, we will try to perform growth rate analysis of various AI/ML projects by considering different types of contributions like pull requests, commits, messages, issues, reviews etc. We will first start by fetching the contribution data for various types and merge them into a singl dataset and in the later part, we will apply the formulas or the models described in this [document](https://docs.google.com/document/d/1ZkPCLNq5UBHrhTNIgAta9cFqevk2rVZ5Vxl9jNuRbQc/edit?usp=sharing).

In [41]:
# importing the required libraries

import json
import pandas as pd
from sqlalchemy import create_engine, text
import plotly.express as px
import plotly.graph_objs as go
import numpy as np
from utils.growth_rate_utils import calculate_log_monthly_growth

## Connect to the Augur database

In [42]:
# Opening the JSON file containing database credentials and loading it into a dictionary
with open("data/il_ai_creds.json") as config_file:
    config = json.load(config_file)
    
# Creating a PostgreSQL database connection string using the credentials from the JSON file
database_connection_string = 'postgresql+psycopg2://{}:{}@{}:{}/{}'.format(
    config['user'],        # Username
    config['password'],    # Password
    config['host'],        # Hostname
    config['port'],        # Port number
    config['database']     # Database name
)

# Assigning the connection string to a variable
connection_string = database_connection_string

# Creating a SQLAlchemy engine using the connection string
engine = create_engine(connection_string)

## Load the URLs of AI repositories

Retrieve the list of repositories that will used for the growth rate analysis

In [43]:
# Opening the JSON file containing AI repository data and loading it into a dictionary
f = open('ai_repos.json')
urls_json_data = json.load(f)

# Closing the file after loading the data
f.close()  

# print the collected repositories
from pprint import pprint
pprint(urls_json_data)


{'gen_ai': ['https://github.com/lucidrains/imagen-pytorch',
            'https://github.com/langchain-ai/langchain',
            'https://github.com/run-llama/llama_index',
            'https://github.com/microsoft/lora',
            'https://github.com/nvidia/nemo',
            'https://github.com/huggingface/peft',
            'https://github.com/microsoft/semantic-kernel',
            'https://github.com/chroma-core/chroma',
            'https://github.com/milvus-io/milvus',
            'https://github.com/qdrant/qdrant',
            'https://github.com/bigscience-workshop/promptsource',
            'https://github.com/automatic1111/stable-diffusion-webui'],
 'llm': ['https://github.com/huggingface/transformers',
         'https://github.com/huggingface/datasets',
         'https://github.com/huggingface/trl',
         'https://github.com/microsoft/deepspeed',
         'https://github.com/timdettmers/bitsandbytes',
         'https://github.com/mistralai/mistral-common',
         'ht

In [44]:
# Initializing an empty list to store repository git URLs
repo_git_set = []

# Extracting the list of repositories from the loaded JSON data
for key in urls_json_data.keys():
    repo_git_set.extend(urls_json_data.get(key))

## Retrieve the repository IDs and the repository names

Let's retrieve the repository IDs and names from the augur database.

In [45]:
# Initializing empty lists to store repository IDs and names
repo_set = []
repo_name_set = []

# Iterating through the list of repository git URLs
for repo_git in repo_git_set:
    # Creating a SQL query to fetch repository ID and name for each git URL
    repo_query = text(f"""
                    SET SCHEMA 'augur_data';
                    SELECT 
                        b.repo_id,
                        b.repo_name
                    FROM
                        repo_groups a,
                        repo b
                    WHERE
                        a.repo_group_id = b.repo_group_id AND
                        b.repo_git = '{repo_git}'
            """)

    # Using the connection to execute the query
    with engine.connect() as connection:
        t = connection.execute(repo_query)  # Executing the query
        results = t.mappings().all()  # Fetching all the results
        
        # Checking if results are found and extracting repo_id and repo_name
        if results:
            repo_id = results[0]['repo_id']
            repo_name = results[0]['repo_name']
        else:
            repo_id = None
            repo_name = None
        
        # Appending the fetched repository ID and name to the respective lists
        repo_set.append(repo_id)
        repo_name_set.append(repo_name)

# Printing the lists of repository IDs and names
print(repo_set)
print(repo_name_set)

[25495, 25498, 25497, 25501, 25500, 25504, 25503, 25557, 25502, 25499, 25511, 25514, 25515, 25505, 25512, 25516, 25507, 25506, 25510, 25509, 25508, 25513, 25518, 25519, 25523, 25522, 25520, 25521, 25517, 25511, 25533, 25525, 25530, 25524, 25528, 25532, 25529, 25481, 25527, 25543, 25541, 25537, 25546, 25534, 25540, 25538, 25545, 25535, 25542, 25536, 25539]
['numpy', 'tensorflow', 'networkx', 'pytorch', 'keras-io', 'tinygrad', 'pandas', 'polars', 'arrow', 'mlx', 'transformers', 'spacy', 'nltk', 'allennlp', 'gensim', 'corenlp', 'deepspeech', 'fasttext', 'sentence-transformers', 'opennmt', 'opennlp', 'cogcomp-nlp', 'mycroft-core', 'open-assistant', 'rhasspy', 'ovos-core', 'jarvis', 'leon', 'porcupine', 'transformers', 'datasets', 'trl', 'deepspeed', 'bitsandbytes', 'mistral-common', 'llama', 'text-to-text-transfer-transformer', 'instructlab', 'gemma', 'imagen-pytorch', 'langchain', 'llama_index', 'lora', 'nemo', 'peft', 'semantic-kernel', 'chroma', 'milvus', 'qdrant', 'promptsource', 'stab

In [46]:
# Creating a dictionary using zip to pair repo_set (IDs) and repo_name_set (names)
repo_dict = dict(zip(repo_set, repo_name_set))

# Printing the dictionary
print(repo_dict)

{25495: 'numpy', 25498: 'tensorflow', 25497: 'networkx', 25501: 'pytorch', 25500: 'keras-io', 25504: 'tinygrad', 25503: 'pandas', 25557: 'polars', 25502: 'arrow', 25499: 'mlx', 25511: 'transformers', 25514: 'spacy', 25515: 'nltk', 25505: 'allennlp', 25512: 'gensim', 25516: 'corenlp', 25507: 'deepspeech', 25506: 'fasttext', 25510: 'sentence-transformers', 25509: 'opennmt', 25508: 'opennlp', 25513: 'cogcomp-nlp', 25518: 'mycroft-core', 25519: 'open-assistant', 25523: 'rhasspy', 25522: 'ovos-core', 25520: 'jarvis', 25521: 'leon', 25517: 'porcupine', 25533: 'datasets', 25525: 'trl', 25530: 'deepspeed', 25524: 'bitsandbytes', 25528: 'mistral-common', 25532: 'llama', 25529: 'text-to-text-transfer-transformer', 25481: 'instructlab', 25527: 'gemma', 25543: 'imagen-pytorch', 25541: 'langchain', 25537: 'llama_index', 25546: 'lora', 25534: 'nemo', 25540: 'peft', 25538: 'semantic-kernel', 25545: 'chroma', 25535: 'milvus', 25542: 'qdrant', 25536: 'promptsource', 25539: 'stable-diffusion-webui'}


Let's convert the data type of repo_set from a list to a tuple so that we can easily pass this in the sql queries.

In [47]:
repo_set_tuple = tuple(repo_set)

In [48]:
# Define a function to execute SQL queries and return a DataFrame
def execute_query(query, engine):
    with engine.connect() as connection:
        result = connection.execute(query)
        return pd.DataFrame(result.fetchall(), columns=result.keys())

## Fetch the contribution data

### Pull Request Contribution data

Let's get the count of pull requests for each repository in `repo_set`, grouped by the year and month of creation, and store the result in a pandas dataframe for further analysis.

In [49]:
# Query to fetch pull request data for repo_ids in repo_set
pull_requests_query = text(f"""
    SELECT 
        repo_id, 
        CAST(DATE_PART('year', pr_created_at) AS INTEGER) AS year,
        CAST(DATE_PART('month', pr_created_at) AS INTEGER) AS month,
        COUNT(pull_request_id) AS pull_request_count
    FROM 
        augur_data.pull_requests
    WHERE 
        repo_id IN :repo_set_tuple
        AND pr_created_at IS NOT NULL 
    GROUP BY 
        repo_id, year, month
    ORDER BY 
        repo_id, year, month;
""")

pull_requests_df = execute_query(pull_requests_query.bindparams(repo_set_tuple=repo_set_tuple), engine)

In [50]:
pull_requests_df

,repo_id,year,month,pull_request_count
0,25481,2024,2,93
1,25481,2024,3,390
2,25481,2024,4,183
3,25481,2024,5,120
4,25481,2024,6,180
...,...,...,...,...
2998,25557,2024,6,370
2999,25557,2024,7,301
3000,25557,2024,8,268
3001,25557,2024,9,258


### Commit contribution data

Let's get the count of commits for each repository in `repo_set`, grouped by the year and month of creation, and store the result in a pandas dataframe for further analysis.

In [51]:
# Query to fetch commits data for repo_ids in repo_set
commits_query = text(f"""
    SELECT 
        repo_id, 
        CAST(DATE_PART('year', cmt_author_timestamp) AS INTEGER) AS year,
        CAST(DATE_PART('month', cmt_author_timestamp) AS INTEGER) AS month,
        COUNT(cmt_id) AS commit_count
    FROM 
        augur_data.commits
    WHERE 
        repo_id IN :repo_set_tuple
        AND cmt_author_timestamp IS NOT NULL 
    GROUP BY 
        repo_id, year, month
    ORDER BY 
        repo_id, year, month;
""")

commits_df = execute_query(commits_query.bindparams(repo_set_tuple=repo_set_tuple), engine)

In [52]:
commits_df

,repo_id,year,month,commit_count
0,25481,2024,2,346
1,25481,2024,3,699
2,25481,2024,4,632
3,25481,2024,5,384
4,25481,2024,6,807
...,...,...,...,...
3533,25557,2024,6,3115
3534,25557,2024,7,1616
3535,25557,2024,8,2080
3536,25557,2024,9,2246


### Issue contribution data

Let's get the count of issues for each repository in `repo_set`, grouped by the year and month of creation, and store the result in a pandas dataframe for further analysis.

In [53]:
# Query to fetch issues data for repo_ids in repo_set
issues_query = text(f"""
    SELECT 
        repo_id, 
        CAST(DATE_PART('year', created_at) AS INTEGER) AS year,
        CAST(DATE_PART('month', created_at) AS INTEGER) AS month,
        
        COUNT(issue_id) AS issue_count
    FROM 
        augur_data.issues
    WHERE 
        repo_id IN :repo_set_tuple
        AND created_at IS NOT NULL 
    GROUP BY 
        repo_id, year, month
    ORDER BY 
        repo_id, year, month;
""")

issues_df = execute_query(issues_query.bindparams(repo_set_tuple=repo_set_tuple), engine)

In [54]:
issues_df

,repo_id,year,month,issue_count
0,25481,2024,2,63
1,25481,2024,3,216
2,25481,2024,4,93
3,25481,2024,5,60
4,25481,2024,6,107
...,...,...,...,...
3002,25557,2024,5,307
3003,25557,2024,6,305
3004,25557,2024,7,354
3005,25557,2024,8,264


### Pull Request Review contribution data

Let's get the count of pull request reviews for each repository in `repo_set`, grouped by the year and month of creation, and store the result in a pandas dataframe for further analysis.

In [55]:
# Query to fetch pull request reviews data for repo_ids in repo_set
pr_reviews_query = text(f"""
    SELECT 
        repo_id, 
        CAST(DATE_PART('year', pr_review_submitted_at) AS INTEGER) AS year,
        CAST(DATE_PART('month', pr_review_submitted_at) AS INTEGER) AS month,
        COUNT(pr_review_id) AS review_count
    FROM 
        augur_data.pull_request_reviews
    WHERE 
        repo_id IN :repo_set_tuple
        AND pr_review_submitted_at IS NOT NULL
    GROUP BY 
        repo_id, year, month
    ORDER BY 
        repo_id, year, month;
""")

pr_reviews_df = execute_query(pr_reviews_query.bindparams(repo_set_tuple=repo_set_tuple), engine)

In [56]:
pr_reviews_df

,repo_id,year,month,review_count
0,25481,2024,2,133
1,25481,2024,3,1173
2,25481,2024,4,614
3,25481,2024,5,581
4,25481,2024,6,1040
...,...,...,...,...
1868,25557,2024,4,374
1869,25557,2024,5,428
1870,25557,2024,6,387
1871,25557,2024,7,420


### Message contribution data

Let's get the count comments or messages for each repository in `repo_set`, grouped by the year and month of creation, and store the result in a pandas dataframe for further analysis.

In [57]:
# Query to fetch messages data for repo_ids in repo_set
messages_query = text(f"""
    SELECT 
        repo_id, 
        CAST(DATE_PART('year', msg_timestamp) AS INTEGER) AS year,
        CAST(DATE_PART('month', msg_timestamp) AS INTEGER) AS month,
        COUNT(msg_id) AS message_count
    FROM 
        augur_data.message
    WHERE 
        repo_id IN :repo_set_tuple
        AND msg_timestamp IS NOT NULL 
    GROUP BY 
        repo_id, year, month
    ORDER BY 
        repo_id, year, month;
""")

messages_df = execute_query(messages_query.bindparams(repo_set_tuple=repo_set_tuple), engine)

In [58]:
messages_df

,repo_id,year,month,message_count
0,25481,2024,2,297
1,25481,2024,3,2233
2,25481,2024,4,1307
3,25481,2024,5,1095
4,25481,2024,6,1733
...,...,...,...,...
3165,25557,2024,5,1775
3166,25557,2024,6,1764
3167,25557,2024,7,1769
3168,25557,2024,8,1555


In [59]:
# Merge all dataframes into one final dataframe

final_df = pull_requests_df.merge(commits_df, on=['repo_id', 'year', 'month'], how='outer')
final_df = final_df.merge(issues_df, on=['repo_id', 'year', 'month'], how='outer')
final_df = final_df.merge(pr_reviews_df, on=['repo_id', 'year', 'month'], how='outer')
final_df = final_df.merge(messages_df, on=['repo_id', 'year', 'month'], how='outer')

In [60]:
final_df

,repo_id,year,month,pull_request_count,commit_count,issue_count,review_count,message_count
0,25481,2024,2,93.0,346.0,63.0,133.0,297.0
1,25481,2024,3,390.0,699.0,216.0,1173.0,2233.0
2,25481,2024,4,183.0,632.0,93.0,614.0,1307.0
3,25481,2024,5,120.0,384.0,60.0,581.0,1095.0
4,25481,2024,6,180.0,807.0,107.0,1040.0,1733.0
...,...,...,...,...,...,...,...,...
3844,25557,2024,6,370.0,3115.0,305.0,387.0,1764.0
3845,25557,2024,7,301.0,1616.0,354.0,420.0,1769.0
3846,25557,2024,8,268.0,2080.0,264.0,281.0,1555.0
3847,25557,2024,9,258.0,2246.0,251.0,NaN,542.0


In [61]:
# Check for missing values in the final dataframe
missing_values = final_df.isnull().sum()
missing_values

repo_id                  0
year                     0
month                    0
pull_request_count     846
commit_count           311
issue_count            842
review_count          1976
message_count          679
dtype: int64

In [62]:
# Fill missing values with 0 and convert to integers
count_columns = ['pull_request_count', 'commit_count', 'issue_count', 'review_count', 'message_count']
final_df[count_columns] = final_df[count_columns].fillna(0).astype(int)

In [63]:
final_df['total_contributions'] = (
    final_df['pull_request_count'] +
    final_df['commit_count'] +
    final_df['issue_count'] +
    final_df['review_count'] +
    final_df['message_count']
)

In [64]:
# Print the final dataframe
final_df.head()

,repo_id,year,month,pull_request_count,commit_count,issue_count,review_count,message_count,total_contributions
0,25481,2024,2,93,346,63,133,297,932
1,25481,2024,3,390,699,216,1173,2233,4711
2,25481,2024,4,183,632,93,614,1307,2829
3,25481,2024,5,120,384,60,581,1095,2240
4,25481,2024,6,180,807,107,1040,1733,3867


## Calculate the Age of Projects

Let's calculate the age of each project. 

The project age is calculated by determining the total number of months between the current date and the date of the earliest recorded commit in the repository. This is done by converting both dates into total months and then finding their difference.

In [65]:
# Query to fetch project age in months for each repository
project_age_query = text(f"""
    SELECT 
        repo_id,
        (DATE_PART('year', NOW()) * 12 + DATE_PART('month', NOW())) - 
        (DATE_PART('year', MIN(cmt_author_timestamp)) * 12 + DATE_PART('month', MIN(cmt_author_timestamp))) 
        AS project_age_in_months
    FROM 
        augur_data.commits
    WHERE 
        repo_id IN :repo_set_tuple AND 
        cmt_author_timestamp IS NOT NULL 
    GROUP BY 
        repo_id;
""")

# Execute the query and load the data into a pandas dataframe
project_age_df = execute_query(project_age_query.bindparams(repo_set_tuple=repo_set_tuple), engine)
project_age_df

,repo_id,project_age_in_months
0,25481,8.0
1,25495,274.0
2,25497,231.0
3,25498,107.0
4,25499,11.0
5,25500,53.0
6,25501,153.0
7,25502,124.0
8,25503,183.0
9,25504,48.0


In [66]:
# Merge project age data with the final dataframe on 'repo_id'
final_df = final_df.merge(project_age_df, on='repo_id', how='left')

In [67]:
final_df

,repo_id,year,month,pull_request_count,commit_count,issue_count,review_count,message_count,total_contributions,project_age_in_months
0,25481,2024,2,93,346,63,133,297,932,8.0
1,25481,2024,3,390,699,216,1173,2233,4711,8.0
2,25481,2024,4,183,632,93,614,1307,2829,8.0
3,25481,2024,5,120,384,60,581,1095,2240,8.0
4,25481,2024,6,180,807,107,1040,1733,3867,8.0
...,...,...,...,...,...,...,...,...,...,...
3844,25557,2024,6,370,3115,305,387,1764,5941,53.0
3845,25557,2024,7,301,1616,354,420,1769,4460,53.0
3846,25557,2024,8,268,2080,264,281,1555,4448,53.0
3847,25557,2024,9,258,2246,251,0,542,3297,53.0


Let's add the repository or project names to the dataframe for better understanding and better analysis.

In [68]:
# Create the 'repo_name' column by mapping 'repo_id' to 'repo_dict'
final_df['repo_name'] = final_df['repo_id'].map(repo_dict)

# Insert 'repo_name' next to 'repo_id'
repo_id_index = final_df.columns.get_loc('repo_id')  # Get the index of 'repo_id' column
final_df.insert(repo_id_index + 1, 'repo_name', final_df.pop('repo_name'))  # Insert 'repo_name' after 'repo_id'

Let's compare the values in repo_name from the growth_df with [ai_repos.json](ai_repos.json) and add a `category` column to the final_df.

In [69]:
# Create a dictionary to map repo names to categories
repo_category_dict = {}

# Map each repo_name in growth_df to its category
for category, repo_list in urls_json_data.items():
    for repo_url in repo_list:
        # Extract repo name from the GitHub URLs
        repo_name = repo_url.split('/')[-1]
        repo_category_dict[repo_name] = category

final_df['category'] = final_df['repo_name'].map(repo_category_dict)

In [70]:
# Display updated dataframe with repo_name and category columns
final_df

,repo_id,repo_name,year,month,pull_request_count,commit_count,issue_count,review_count,message_count,total_contributions,project_age_in_months,category
0,25481,instructlab,2024,2,93,346,63,133,297,932,8.0,llm
1,25481,instructlab,2024,3,390,699,216,1173,2233,4711,8.0,llm
2,25481,instructlab,2024,4,183,632,93,614,1307,2829,8.0,llm
3,25481,instructlab,2024,5,120,384,60,581,1095,2240,8.0,llm
4,25481,instructlab,2024,6,180,807,107,1040,1733,3867,8.0,llm
...,...,...,...,...,...,...,...,...,...,...,...,...
3844,25557,polars,2024,6,370,3115,305,387,1764,5941,53.0,math
3845,25557,polars,2024,7,301,1616,354,420,1769,4460,53.0,math
3846,25557,polars,2024,8,268,2080,264,281,1555,4448,53.0,math
3847,25557,polars,2024,9,258,2246,251,0,542,3297,53.0,math


In [71]:
# Let's create a deep copy of the final_df to avoid modifying the original DataFrame
data = final_df.copy()

## Growth Rate Calculation

Let's calculate the growth rate of each repository for the first and last 6 months.

- We are calculating the growth rate by taking the logarithmic growth rate between consecutive months 
- The log transformation captures relative (percentage) changes rather than absolute changes. This makes it easier to compare growth rates across repositories, even if they have vastly different numbers of total contributions.
- Logarithmic growth rates reduce the impact of very large outliers, smoothing out extreme spikes or drops in the data.

**Calculation:**
- `np.log(df['total_contributions'])` takes the natural logarithm of the total_contributions for each month
- `df['total_contributions'].shift(1)` The shift(1) method shifts the total_contributions data by 1 row, effectively aligning the value from the previous month with the current month. This allows us to compare the current month's contributions with the previous month's.
- `np.log(df['total_contributions']) - np.log(df['total_contributions'].shift(1))`: This calculates the difference in the logarithms of the contributions between the current month and the previous month. In logarithmic terms, this difference represents the logarithmic growth rate or log-return between the two periods.
- The result is multiplied by 100 to express the logarithmic growth rate as a percentage

In [72]:
# Sort the data by repo_id, year, and month to ensure chronological order
data = data.sort_values(by=['repo_id', 'year', 'month'])

### Last 6 month's contributions

In [73]:
# Function to filter and calculate log growth rate for the last 6 months of each repo

def get_last_6_months_growth(df, months_to_consider=7):
    
    # Sort by year and month to ensure chronological order
    df = df.sort_values(by=['year', 'month'])
    
    # Take the last 'months_to_consider' months
    if len(df) >= months_to_consider:
        df = df.tail(months_to_consider)
    else:
        return pd.DataFrame()  # Return empty DataFrame if less than required months

    # Apply the growth rate calculation for the last 6 months
    df = calculate_log_monthly_growth(df)
    
    return df

# Apply the function to each repository group to get the last 6 months of data
last_6months_df = data.groupby('repo_id').apply(get_last_6_months_growth)

# Flatten the groupby result and reset the index
last_6months_df = last_6months_df.reset_index(drop=True)

# Reorder the columns as needed
last_6months_df = last_6months_df[['repo_id', 'repo_name', 'year', 'month', 'x_axis',
                       'pull_request_count', 'commit_count', 'issue_count', 
                       'review_count', 'message_count', 'total_contributions', 'category',
                       'log_growth_rate']]

# Display or print the final dataframe for the last 6 months
last_6months_df

/var/folders/7w/mm9ktp7n75dgn8rjth6_zv_m0000gn/T/ipykernel_68848/3341331297.py:20: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,repo_id,repo_name,year,month,x_axis,pull_request_count,commit_count,issue_count,review_count,message_count,total_contributions,category,log_growth_rate
0,25481.0,instructlab,2024.0,5.0,1.0,120.0,384.0,60.0,581.0,1095.0,2240.0,llm,-23.344743
1,25481.0,instructlab,2024.0,6.0,2.0,180.0,807.0,107.0,1040.0,1733.0,3867.0,llm,54.600315
2,25481.0,instructlab,2024.0,7.0,3.0,261.0,1149.0,143.0,2220.0,3294.0,7067.0,llm,60.295705
3,25481.0,instructlab,2024.0,8.0,4.0,149.0,387.0,74.0,807.0,1523.0,2940.0,llm,-87.702648
4,25481.0,instructlab,2024.0,9.0,5.0,92.0,323.0,73.0,0.0,484.0,972.0,llm,-110.680906
...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,25557.0,polars,2024.0,6.0,2.0,370.0,3115.0,305.0,387.0,1764.0,5941.0,math,22.752952
290,25557.0,polars,2024.0,7.0,3.0,301.0,1616.0,354.0,420.0,1769.0,4460.0,math,-28.672870
291,25557.0,polars,2024.0,8.0,4.0,268.0,2080.0,264.0,281.0,1555.0,4448.0,math,-0.269421
292,25557.0,polars,2024.0,9.0,5.0,258.0,2246.0,251.0,0.0,542.0,3297.0,math,-29.944159


In [74]:
# Function to filter and calculate log growth rate for the first 6 months of each repo
def get_first_6_months_growth(df, months_to_consider=7):
    # Sort by year and month to ensure chronological order
    df = df.sort_values(by=['year', 'month'])

    # Take the first 'months_to_consider' months since the start of the project
    if len(df) >= months_to_consider:
        df = df[:months_to_consider]
    else:
        return pd.DataFrame()  # Return empty DataFrame if less than required months

    # Apply the growth rate calculation for the first 6 months
    df = calculate_log_monthly_growth(df)
    
    return df

# Apply the function to each repository group to get the first 6 months of data
first_6months_df = data.groupby('repo_id').apply(get_first_6_months_growth)

# Flatten the groupby result and reset the index
first_6months_df = first_6months_df.reset_index(drop=True)

# Add repo_name using the existing repo_dict
first_6months_df['repo_name'] = first_6months_df['repo_id'].map(repo_dict)

# Reorder the columns as needed
first_6months_df = first_6months_df[['repo_id', 'repo_name', 'year', 'month', 'x_axis',
                       'pull_request_count', 'commit_count', 'issue_count', 
                       'review_count', 'message_count', 'total_contributions', 'category',
                       'log_growth_rate']]

# Display or print the final dataframe for the first 6 months
first_6months_df

/var/folders/7w/mm9ktp7n75dgn8rjth6_zv_m0000gn/T/ipykernel_68848/2108943184.py:18: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,repo_id,repo_name,year,month,x_axis,pull_request_count,commit_count,issue_count,review_count,message_count,total_contributions,category,log_growth_rate
0,25481.0,instructlab,2024.0,3.0,1.0,390.0,699.0,216.0,1173.0,2233.0,4711.0,llm,162.032266
1,25481.0,instructlab,2024.0,4.0,2.0,183.0,632.0,93.0,614.0,1307.0,2829.0,llm,-50.997691
2,25481.0,instructlab,2024.0,5.0,3.0,120.0,384.0,60.0,581.0,1095.0,2240.0,llm,-23.344743
3,25481.0,instructlab,2024.0,6.0,4.0,180.0,807.0,107.0,1040.0,1733.0,3867.0,llm,54.600315
4,25481.0,instructlab,2024.0,7.0,5.0,261.0,1149.0,143.0,2220.0,3294.0,7067.0,llm,60.295705
...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,25557.0,polars,2020.0,7.0,2.0,0.0,448.0,24.0,0.0,18.0,490.0,math,-11.369305
290,25557.0,polars,2020.0,8.0,3.0,5.0,368.0,23.0,0.0,13.0,409.0,math,-18.069024
291,25557.0,polars,2020.0,9.0,4.0,6.0,493.0,22.0,1.0,32.0,554.0,math,30.344953
292,25557.0,polars,2020.0,10.0,5.0,3.0,592.0,25.0,5.0,64.0,689.0,math,21.807658


In [75]:
import pandas as pd

# Create a new Excel file and write both DataFrames to separate sheets
with pd.ExcelWriter('growth_rate.xlsx') as writer:
    first_6months_df.to_excel(writer, sheet_name='First 6 Months', index=False)
    last_6months_df.to_excel(writer, sheet_name='Last 6 Months', index=False)

### Growth Rate Analysis - Plots

Let's try to create growth rate plots for the projects based on their category. In order to plot based on the category.

In [76]:
# Function to plot growth rate by category, using the custom x-axis
def plot_growth_rate_by_category(df, category_name, title):
    # Filter by category
    category_df = df[df['category'] == category_name]
    
    # Create line plot for growth rate trends over time using 'x_axis' as the x-axis
    fig = px.line(category_df, x='x_axis', y='log_growth_rate', color='repo_name', 
                  title=f'{title} for {category_name} repositories', markers=True)
    
    fig.update_layout(xaxis_title="Time (in months)", yaxis_title="Growth Rate",
                      xaxis=dict(tickvals=[1, 2, 3, 4, 5, 6], ticktext=['1', '2', '3', '4', '5', '6']),
                      legend_title="Repository", title_x=0.5)
    
    fig.show()

In [77]:
# Fetch the categories from ai_repos.json
categories = list(urls_json_data.keys())

# Plot for each individual category
for category in categories:
    plot_growth_rate_by_category(last_6months_df, category, title="Last 6 Months Growth Rate Trend")


In [78]:
# Fetch the categories from ai_repos.json
categories = list(urls_json_data.keys())

# Plot for each individual category
for category in categories:
    plot_growth_rate_by_category(first_6months_df, category, title="First 6 Months Growth Rate Trend")